# Getting Familiar with Plotly

[Plotly](https://plotly.com/python/) is a Python graphing library for building **interactive** charts —
you can hover over points for tooltips, zoom, pan, and toggle legend items directly in the output, no
extra code required.

There are two APIs:

- **Plotly Express** (`plotly.express`, usually imported as `px`) — a high-level API where most charts
  are a single function call. This is what we'll use today.
- **Graph Objects** (`plotly.graph_objects`, usually imported as `go`) — a lower-level API for full
  control over every trace. We'll peek at it briefly at the end.

We'll work with the **gapminder** dataset, which ships built into Plotly Express (no download needed).
It tracks life expectancy, population, and GDP per capita for many countries over several decades —
a great mix of categorical (`country`, `continent`) and numeric (`year`, `lifeExp`, `pop`, `gdpPercap`)
columns to plot with.

Throughout this notebook, sections marked **Try it** show the same chart redrawn with a different config — the parameters worth changing are marked with `# change me` comments. Edit them and re-run the cell to see the effect.

## 1. Setup

Load the library and preview the dataset.

In [27]:
import plotly.express as px

df = px.data.gapminder()
df.head()

,country,continent,year,lifeExp,pop,gdpPercap,iso_alpha,iso_num
0,Afghanistan,Asia,1952,28.801,8425333,779.445314,AFG,4
1,Afghanistan,Asia,1957,30.332,9240934,820.853030,AFG,4
2,Afghanistan,Asia,1962,31.997,10267083,853.100710,AFG,4
3,Afghanistan,Asia,1967,34.020,11537966,836.197138,AFG,4
4,Afghanistan,Asia,1972,36.088,13079460,739.981106,AFG,4


In [28]:
df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
country,1704,142,Afghanistan,12,NaN,NaN,NaN,NaN,NaN,NaN,NaN
continent,1704,5,Africa,624,NaN,NaN,NaN,NaN,NaN,NaN,NaN
year,1704.0,NaN,NaN,NaN,1979.5,17.26533,1952.0,1965.75,1979.5,1993.25,2007.0
lifeExp,1704.0,NaN,NaN,NaN,59.474439,12.917107,23.599,48.198,60.7125,70.8455,82.603
pop,1704.0,NaN,NaN,NaN,29601212.324531,106157896.743915,60011.0,2793664.0,7023595.5,19585221.75,1318683096.0
gdpPercap,1704.0,NaN,NaN,NaN,7215.327081,9857.454543,241.165876,1202.060309,3531.846989,9325.462346,113523.1329
iso_alpha,1704,141,KOR,24,NaN,NaN,NaN,NaN,NaN,NaN,NaN
iso_num,1704.0,NaN,NaN,NaN,425.880282,248.305709,4.0,208.0,410.0,638.0,894.0


## 2. Scatter plot — encoding multiple variables at once

A scatter plot is the natural starting point in Plotly Express: pass a dataframe and column names, and
`px.scatter` maps them onto x, y, color, and size for you. This is the core idea of Plotly Express —
*describe the mapping, not the drawing*.

Here we look at the most recent year (2007): GDP per capita vs. life expectancy, colored by continent,
sized by population. Hover over any point to see its details.

In [29]:
df_2007 = df[df["year"] == 2007]

fig = px.scatter(
    df_2007,
    x="gdpPercap",
    y="lifeExp",
    color="continent",
    size="pop",
    hover_name="country",
    log_x=True,
    size_max=60,
    title="Life Expectancy vs. GDP per Capita (2007)",
)
fig.show()

### Try it

Same chart, different config — swap the axes, grouping, and year to see how the story changes.

In [30]:
fig = px.scatter(
    df[df["year"] == 1962],       # change me: any year in the dataset
    x="pop",                      # change me: try "gdpPercap" or "lifeExp"
    y="gdpPercap",                # change me: try "lifeExp" or "pop"
    color="country",              # change me: try "continent", or drop this line entirely
    size="pop",
    hover_name="country",
    log_x=True,
    size_max=60,
    title="gdpPercap vs. pop, colored by country (1962)",
)
fig.show()

## 3. Line chart — trends over time

Line charts are ideal for showing change over a continuous axis like time. Let's track life expectancy
for a handful of countries across the full dataset.

In [31]:
countries = ["United States", "China", "India", "Nigeria", "Brazil"]
df_countries = df[df["country"].isin(countries)]

fig = px.line(
    df_countries,
    x="year",
    y="lifeExp",
    color="country",
    markers=True,
    title="Life Expectancy Over Time",
)
fig.show()

### Try it

Same chart, different set of countries.

In [32]:
other_countries = ["Germany", "France", "Japan", "South Africa"]  # change me: any countries in the dataset

fig = px.line(
    df[df["country"].isin(other_countries)],
    x="year",
    y="lifeExp",
    color="country",
    markers=True,
    title="Life Expectancy Over Time (a different set of countries)",
)
fig.show()

## 4. Bar chart — comparing categories

Bar charts compare a numeric value across categories. Here's total population by continent in 2007.

In [33]:
pop_by_continent = df_2007.groupby("continent", as_index=False)["pop"].sum()

fig = px.bar(
    pop_by_continent,
    x="continent",
    y="pop",
    color="continent",
    title="Total Population by Continent (2007)",
)
fig.show()

### Try it

Same chart, a different year — compare how the population split has shifted.

In [34]:
year = 1962  # change me: any year in the dataset

pop_by_continent_alt = df[df["year"] == year].groupby("continent", as_index=False)["pop"].sum()

fig = px.bar(
    pop_by_continent_alt,
    x="continent",
    y="pop",
    color="continent",
    title=f"Total Population by Continent ({year})",
)
fig.show()

## 5. Distributions — histogram and box plot

Plotly Express also covers statistical charts. A histogram shows the shape of a distribution; a box
plot summarizes it (median, quartiles, outliers) and makes it easy to compare across groups.

In [35]:
fig = px.histogram(
    df_2007,
    x="lifeExp",
    color="continent",
    nbins=30,
    title="Distribution of Life Expectancy (2007)",
)
fig.show()

In [36]:
fig = px.box(
    df_2007,
    x="continent",
    y="lifeExp",
    color="continent",
    points="all",
    title="Life Expectancy by Continent (2007)",
)
fig.show()

## 6. Animation — Plotly's signature feature

Adding `animation_frame` turns any scatter plot into a playable animation with almost no extra code.
This is the famous "gapminder bubble chart" — press play and watch the world develop over 55 years.

Note `range_x` / `range_y` are fixed so the axes don't jump around between frames, and
`animation_group="country"` keeps each bubble tracking the same country as it moves.

In [37]:
fig = px.scatter(
    df,
    x="gdpPercap",
    y="lifeExp",
    animation_frame="year",
    animation_group="country",
    color="continent",
    size="pop",
    hover_name="country",
    log_x=True,
    size_max=60,
    range_x=[100, 100000],
    range_y=[20, 90],
    title="Life Expectancy vs. GDP per Capita Over Time",
)
fig.show()

## 7. Choropleth map — geographic data

For data with a country/region dimension, a choropleth colors each region by value. Plotly Express
recognizes ISO-3 country codes (the `iso_alpha` column here) and handles the map projection for you.

In [38]:
fig = px.choropleth(
    df,
    locations="iso_alpha",
    color="lifeExp",
    hover_name="country",
    animation_frame="year",
    color_continuous_scale=px.colors.sequential.Plasma,
    title="Life Expectancy by Country Over Time",
)
fig.show()

## 8. Customizing figures

Every `px` function returns a `Figure` object you can keep tweaking with `update_layout` (overall
layout: titles, fonts, legend) and `update_traces` (per-trace appearance: marker style, line width,
opacity).

In [39]:
fig = px.bar(
    pop_by_continent,
    x="continent",
    y="pop",
    title="Total Population by Continent (2007)",
)

fig.update_traces(marker_color="teal", marker_line_color="black", marker_line_width=1, opacity=0.85)
fig.update_layout(
    template="plotly_white",
    title_font_size=20,
    xaxis_title="Continent",
    yaxis_title="Population",
)
fig.show()

## Next steps

- Browse the full chart gallery: https://plotly.com/python/
- Explore the `graph_objects` API (`go.Figure`, `go.Scatter`, ...) when you need finer control than
  `px` gives you — every `px` figure is actually built from `go` traces under the hood.
- This repo is set up for **Dash** too — Dash lets you wrap Plotly figures in an interactive web app
  with dropdowns, sliders, and callbacks, turning a notebook chart into a shareable tool.